б)


In [1]:
import numpy as np
import scipy.stats as st
from scipy.optimize import minimize

In [ ]:
freqs = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])
n = 100
values = np.arange(10)
alpha = 0.05

sample = np.repeat(values, freqs) 


intervals = [-np.inf, 1, 2, 3, 4, 5, 6, 7, 8, 9, np.inf]

# КРИТЕРИЙ ПИРСОНА


def neg_log_likelihood(params):
    theta1, theta2 = params 
    if theta2 <= 0: return np.inf 
    
    
    probs = np.diff(st.norm.cdf(intervals, loc=theta1, scale=theta2))
    
    probs = np.clip(probs, 1e-10, 1.0) 
    
    
    return -np.sum(freqs * np.log(probs))

# Черновая оценка 
theta1_start = np.mean(sample)
theta2_start = np.std(sample, ddof=1)

# точные оценки (ОМПГ)
res = minimize(neg_log_likelihood, [theta1_start, theta2_start], method='Nelder-Mead')
theta1_ompg, theta2_ompg = res.x

print(f"Оценки ОМПГ: theta1  = {theta1_ompg:.4f}, theta2  = {theta2_ompg:.4f}")


probs_ompg = np.diff(st.norm.cdf(intervals, loc=theta1_ompg, scale=theta2_ompg))
expected_freqs = n * probs_ompg

print(f" частоты: \n{np.round(expected_freqs, 2)}")



delta_pearson = np.sum((freqs - expected_freqs)**2 / expected_freqs)


df = len(freqs) - 2 - 1
p_val_pearson = 1 - st.chi2.cdf(delta_pearson, df)

print(f"Delta Пирсона : {delta_pearson:.4f}")
print(f"хи - квадрат( {df})")
print(f"P-value Пирсона: {p_val_pearson:.4f}")

if p_val_pearson < alpha:
    print("отвергаем H0")
else:
    print("Нет оснований отвергнуть H0")


Оценки ОМПГ: theta1  = 5.2897, theta2  = 2.6795
 частоты: 
[ 5.47  5.51  8.66 11.87 14.18 14.76 13.38 10.58  7.28  8.31]
Delta Пирсона : 9.8026
хи - квадрат( 7)
P-value Пирсона: 0.2000
Нет оснований отвергнуть H0


In [4]:
# КРИТЕРИЙ КОЛМОГОРОВА



theta1_hat = np.mean(sample)
theta2_hat = np.std(sample, ddof=1)


D_real = st.kstest(sample, 'norm', args=(theta1_hat, theta2_hat)).statistic
delta_kolm_real = np.sqrt(n) * D_real 

print(f"статистика (дельта) Колмогорова : {delta_kolm_real:.4f}")


N_boot = 50000
count_larger = 0 

for _ in range(N_boot):
    
    fake_sample = np.random.normal(loc=theta1_hat, scale=theta2_hat, size=n)
    
    
    theta1_fake = np.mean(fake_sample)
    theta2_fake = np.std(fake_sample, ddof=1)
    
    
    D_fake = st.kstest(fake_sample, 'norm', args=(theta1_fake, theta2_fake)).statistic
    delta_fake = np.sqrt(n) * D_fake
    
    
    if delta_fake >= delta_kolm_real:
        count_larger += 1


p_val_kolm = count_larger / N_boot

print(f"(l): {count_larger} из {N_boot}")
print(f"P-value Колмогорова): {p_val_kolm:.4f}")


if  p_val_kolm < alpha:
    print("отвергаем H0")
else:
    print("Нет оснований отвергнуть H0")

статистика (дельта) Колмогорова : 1.0021
(l): 698 из 50000
P-value Колмогорова): 0.0140
отвергаем H0
